# Flood Forecasting Workflow

End-to-end flood risk assessment using AquaScope's challenge framework.

**Pipeline:** Data → Model → Forecast → Risk Assessment → Visualisation

In [ ]:
import numpy as np, pandas as pd
%matplotlib inline

rng = np.random.default_rng(42)
days = 365 * 8
dates = pd.date_range("2016-01-01", periods=days, freq="D")
base = 30 + 20 * np.sin(2 * np.pi * np.arange(days) / 365)
Q = pd.Series(np.maximum(base + rng.exponential(8, days) + rng.choice([0]*20+[1], days)*rng.exponential(80, days), 1.0), index=dates)
print(f"Period: {dates[0].date()} → {dates[-1].date()}, Max Q = {Q.max():.0f} m³/s")

## Run FloodChallenge

In [ ]:
from aquascope.challenges.flood import FloodChallenge

fc = FloodChallenge()
fc.load_dataframe(Q.to_frame(name="value"))
fc.fit(model_name="arima")
forecast = fc.forecast(days=14)
forecast.head()

In [ ]:
risk = fc.assess_risk(forecast)
print(f"Risk Level: {risk['risk_level']}")
print(f"Description: {risk['description']}")
print(f"Peak Forecast: {risk['peak_forecast']:.1f} m³/s")
print(f"95th Percentile Threshold: {risk['threshold_95']:.1f} m³/s")

## Visualise Results

In [ ]:
from aquascope.viz import plot_forecast, plot_return_periods

# Forecast plot
recent = Q.iloc[-90:].to_frame(name="value")
plot_forecast(observed=recent, forecast=forecast, title="14-Day Flood Forecast", ylabel="Q (m³/s)")

In [ ]:
# Return periods
if risk.get("return_periods"):
    plot_return_periods(risk["return_periods"], observed_max=float(Q.max()), title="Flood Return Periods")

## Using the NL Agent

You can also run this entire workflow with a single command:

```python
from aquascope.ai_engine.agent import HydroAgent
agent = HydroAgent()
result = agent.solve("Forecast flooding at lat 25.03, lon 121.57 for 14 days")
print(agent.explain(result))
```